# 01 — Data Pipeline
Download, clean, and prepare the dataset for the Nexus recommendation system.

**Dataset:** Online Retail II (UCI) — real transaction data with user IDs, item IDs, and purchase history.
We map purchase behavior to our auction interaction schema (view=1, favorite=2, bid=3).

## 1. Install Dependencies

In [ ]:
# Run this cell once
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn sentence-transformers implicit scipy openpyxl kaggle jupyter matplotlib seaborn

## 2. Download Dataset

In [ ]:
import os, urllib.request, zipfile

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

ZIP_PATH = '../data/raw/online_retail.zip'

if not os.path.exists(ZIP_PATH):
    print('Downloading dataset...')
    urllib.request.urlretrieve(
        'https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip',
        ZIP_PATH
    )
    print('Download complete.')
else:
    print('Dataset already downloaded.')

# Extract
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('../data/raw/')
    print('Files extracted:', z.namelist())

## 3. Load Raw Data

In [ ]:
import pandas as pd
import numpy as np

# Load both year sheets and combine
print('Loading Year 2009-2010...')
df1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')

print('Loading Year 2010-2011...')
df2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([df1, df2], ignore_index=True)

print(f'\nRaw shape: {df.shape}')
df.head()

In [ ]:
# Quick overview of raw data
print('Column types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

## 4. Clean Data

In [ ]:
def clean_data(df):
    """
    Clean and map columns to our auction interaction schema:
      user_id, item_id, item_title, interaction_weight, interaction_timestamp
    """
    original_size = len(df)

    # Drop rows missing customer ID or item description
    df = df.dropna(subset=['Customer ID', 'Description'])
    print(f'After dropping nulls: {len(df):,} rows (removed {original_size - len(df):,})')

    # Remove cancelled orders (Invoice starts with 'C')
    df = df[~df['Invoice'].astype(str).str.startswith('C')]
    print(f'After removing cancellations: {len(df):,} rows')

    # Remove zero/negative quantity or price
    df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
    print(f'After removing invalid quantities/prices: {len(df):,} rows')

    # Rename columns to match our schema
    df = df.rename(columns={
        'Customer ID':     'user_id',
        'StockCode':       'item_id',
        'Description':     'item_title',
        'InvoiceDate':     'interaction_timestamp',
        'Quantity':        'quantity',
        'Price':           'price',
    })

    # Map purchase quantity → interaction weight (proxy for view/favorite/bid)
    # High quantity purchase ≈ strong interest (bid=3)
    # Medium quantity ≈ moderate interest (favorite=2)
    # Single purchase ≈ light interest (view=1)
    df['interaction_weight'] = df['quantity'].apply(
        lambda q: 3 if q >= 5 else (2 if q >= 2 else 1)
    )

    # Standardize ID types
    df['user_id'] = df['user_id'].astype(int).astype(str)
    df['item_id'] = df['item_id'].astype(str).str.strip().str.upper()
    df['item_title'] = df['item_title'].astype(str).str.strip().str.lower()

    # Keep only needed columns
    df = df[[
        'user_id', 'item_id', 'item_title',
        'interaction_weight', 'interaction_timestamp',
        'quantity', 'price'
    ]].drop_duplicates()

    return df


clean = clean_data(df)
print(f'\nFinal clean shape: {clean.shape}')
print(f'Unique users:  {clean["user_id"].nunique():,}')
print(f'Unique items:  {clean["item_id"].nunique():,}')
clean.head()

## 5. Explore the Data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Interaction weight distribution
clean['interaction_weight'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#4A90D9', '#F5A623', '#E74C3C']
)
axes[0].set_title('Interaction Weight Distribution')
axes[0].set_xlabel('Weight (1=view, 2=favorite, 3=bid)')
axes[0].set_xticklabels(['1', '2', '3'], rotation=0)

# Interactions per user (log scale)
user_counts = clean.groupby('user_id').size()
axes[1].hist(user_counts, bins=50, color='#4A90D9', edgecolor='white')
axes[1].set_title('Interactions per User')
axes[1].set_xlabel('Number of interactions')
axes[1].set_ylabel('Number of users')
axes[1].set_yscale('log')

# Top 20 most popular items
top_items = clean.groupby('item_title')['interaction_weight'].sum().nlargest(20)
top_items.plot(kind='barh', ax=axes[2], color='#4A90D9')
axes[2].set_title('Top 20 Items by Interaction Score')
axes[2].set_xlabel('Total interaction weight')

plt.tight_layout()
plt.savefig('../data/processed/exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sparsity check — important for collaborative filtering
n_users = clean['user_id'].nunique()
n_items = clean['item_id'].nunique()
n_interactions = len(clean)
sparsity = 1 - (n_interactions / (n_users * n_items))

print(f'Users:        {n_users:,}')
print(f'Items:        {n_items:,}')
print(f'Interactions: {n_interactions:,}')
print(f'Matrix sparsity: {sparsity:.4%}')
print('\n(Typical recommendation matrices are >99% sparse — that is normal)')

## 6. Build Item Catalog

In [ ]:
# Deduplicated item catalog — one row per item
# Used by content-based filtering in notebook 02
catalog = (
    clean.groupby('item_id')
         .agg(
             item_title=('item_title', 'first'),
             avg_price=('price', 'mean'),
             popularity=('interaction_weight', 'sum'),  # γ (popularity score)
             n_interactions=('interaction_weight', 'count'),
         )
         .reset_index()
)
catalog['avg_price'] = catalog['avg_price'].round(2)

print(f'Item catalog size: {len(catalog):,}')
catalog.sort_values('popularity', ascending=False).head(10)

## 7. Train / Test Split

In [ ]:
# Temporal split: train on earlier interactions, test on later ones
# This simulates real-world usage better than random split
clean['interaction_timestamp'] = pd.to_datetime(clean['interaction_timestamp'])
clean = clean.sort_values('interaction_timestamp')

split_idx = int(len(clean) * 0.8)
train = clean.iloc[:split_idx].copy()
test  = clean.iloc[split_idx:].copy()

print(f'Train: {len(train):,} interactions')
print(f'Test:  {len(test):,} interactions')
print(f'Train date range: {train["interaction_timestamp"].min().date()} → {train["interaction_timestamp"].max().date()}')
print(f'Test  date range: {test["interaction_timestamp"].min().date()} → {test["interaction_timestamp"].max().date()}')

## 8. Save Processed Files

In [ ]:
clean.to_csv('../data/processed/interactions.csv', index=False)
catalog.to_csv('../data/processed/item_catalog.csv', index=False)
train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

print('Saved:')
print('  data/processed/interactions.csv  — full cleaned interactions')
print('  data/processed/item_catalog.csv  — deduplicated item catalog')
print('  data/processed/train.csv         — 80% temporal train split')
print('  data/processed/test.csv          — 20% temporal test split')
print('\nData pipeline complete. Run notebook 02 next.')